In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-04-28 15:48:59,362 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-28 15:48:59,363 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
print(len(cultpass_articles))

NameError: name 'cultpass_articles' is not defined

In [7]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [ ]:
# TODO: Create additional 10 articles

# created additional 10 articles in cultpass_articles.jsonl


In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [9]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='cdc47e74-0122-4c8d-b9ba-b514d211b1d7', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='bf6548a6-9c93-4fbb-a474-f8b78ecd916b', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='0055c99e-b6db-4c64-b8be-2edad86a1ae9', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='3f0b68fc-f223-4915-b0b6-c7326b04594f', title='How to Handle Login Issues?')>
<Knowledge(article_id='56e40466-54e7-4728-a4f3-7644b7b2c936', title='Understanding Membership Tiers')>
<Knowledge(article_id='6fb2b7cf-337a-4016-b3cb-6634affdd37b', title='What to Do If an Event Is Fully Booked')>
<Knowledge(article_id='05f907c6-a31d-4d3d-ae91-b5411f190aea', title='How to Check Your Monthly Quota')>
<Knowledge(article_id='181eaaae-023d-4574-8f1f-8e4962e85792', title='Late Arrival Policy for Events')>
<Knowledge(article_id='edd45e05-67e2-4203-aea2-ef137e1b52d8', title='How QR Code Entry Works')>
<Knowledge(article_id='ddb0d04e-d008-42e6-bc1d-0a1c

In [17]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='5d233067-4fa1-476b-a359-387d1366053f', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [18]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='d9795bc5-b347-4014-b863-8f92cfe2c944', role='user', content='I can't log in to my Cultpass ...')>


In [20]:

import sqlite3

conn = sqlite3.connect("data/core/udahub.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS ticket_logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ticket_id TEXT NOT NULL,
    stage TEXT NOT NULL,
    payload TEXT,
    created_at TEXT
)
""")

conn.commit()
conn.close()

print("✅ ticket_logs table created")


✅ ticket_logs table created


In [23]:

import sqlite3

conn = sqlite3.connect("data/core/udahub.db")
cur = conn.cursor()

cur.execute("SELECT ticket_id, stage, payload FROM ticket_logs ORDER BY id DESC LIMIT 10")
rows = cur.fetchall()
conn.close()

rows


[('2',
  'analysis_start',
  '{"intent": "urgent", "ticket_metadata": {"urgency": "high", "complexity": "low", "status": "open", "issue_type": "general"}, "tools_used": [], "history_loaded": 0}'),
 ('2',
  'analysis_start',
  '{"intent": "urgent", "ticket_metadata": {"urgency": "high", "complexity": "low", "status": "open", "issue_type": "general"}, "tools_used": [], "history_loaded": 0}'),
 ('2',
  'analysis_kb_success',
  '{"confidence": 0.878, "intent": "qa", "sources": ["How to Reserve a Spot for an Event", "What to Do If an Event Is Fully Booked", "How to Cancel or Pause a Subscription"], "history_used": 0}'),
 ('2',
  'analysis_start',
  '{"intent": "unknown", "ticket_metadata": {"urgency": "low", "complexity": "low", "status": "open", "issue_type": "general"}, "tools_used": [], "history_loaded": 0}')]